# Optimización Automática de Hiperparámetros con Optuna

Este cuaderno implementa la sintonía fina de hiperparámetros para los modelos del proyecto utilizando **Optuna**, una librería de optimización bayesiana de alto rendimiento.

El algoritmo aprende de las ejecuciones previas (a través de TPE - Tree-structured Parzen Estimator) y descarta de manera temprana pruebas poco prominentes mediante **Poda (Pruning)** para optimizar el tiempo de GPU.

**Nota sobre la partición:** Reservamos de forma fija a `tania` (mujer) y `matias` (hombre), además de los sujetos incompletos (`eduardo`), para conformar el Held-out Test Set. El resto de los 9 sujetos completos se utiliza para la validación cruzada interna (5 folds).

**Nota sobre la reproducibilidad:** Forzamos el reseteo de la semilla global (`set_seed(42)`) al inicio de cada prueba (trial) en la función objetivo. Esto garantiza que todos los modelos comiencen exactamente con los **mismos pesos iniciales**.

In [3]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import optuna
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net import MTDE_Net
from src.models.cnn_lstm import ThermalCNNLSTM
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.loaders.cnn_lstm_loader import ThermalSequenceDataset
from src.utils import (
    SqrtScaledMSELoss,
    eval_mtde_net_metrics,
    eval_cnn_lstm_metrics,
    build_subject_split_plan,
    sequence_ids_for_subjects
)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

d:\ulima\Ulima_archivos\UL-2026-1\Seminario1\experimentacion\flir_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración de Partición General (Plan Híbrido)

In [4]:
metadata_path = "../processed_data/metadata_train.csv"

# Instanciar primero los datasets para que filtren y reinicien sus índices
train_ds_opt = MultimodalThermalDataset(metadata_csv=metadata_path, is_train=True)
val_ds_opt = MultimodalThermalDataset(metadata_csv=metadata_path, is_train=False)
train_ds_opt.root = Path("../processed_data")
val_ds_opt.root = Path("../processed_data")

# Construir el plan de partición híbrido:
# Reservamos a 'tania' (mujer) y 'matias' (hombre) para Test Set, junto con los incompletos ('eduardo')
plan_opt = build_subject_split_plan(
    train_ds_opt.df,
    n_splits=5,
    seed=42,
    test_subjects=["tania", "matias"],
    reserve_incomplete_for_test=True,
)

print("=== PLAN DE PARTICIÓN CONFIGURADO ===")
print(f"Sujetos en Test ({plan_opt.num_test_subjects}): {plan_opt.test_subjects}")
print(f"Sujetos en Train/Val ({plan_opt.num_trainval_subjects}): {plan_opt.trainval_subjects}")
print(f"Folds de validación cruzada: {len(plan_opt.folds)}")

=== PLAN DE PARTICIÓN CONFIGURADO ===
Sujetos en Test (3): ['eduardo', 'matias', 'tania']
Sujetos en Train/Val (9): ['alonso', 'angelo', 'claudia', 'esteban', 'fabricio', 'juandiego', 'melissa', 'renato', 'yohamin']
Folds de validación cruzada: 5


## 2. Optimización para MTDE-Net

In [3]:
def objective_mtde_net(trial):
    set_seed(42)
    
    # Espacio de Búsqueda para MTDE-Net
    lr = trial.suggest_float("lr", 5e-5, 2e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    time_scale = 30.0
    fold_maes = []
    
    for fold_idx, fold in enumerate(plan_opt.folds):
        train_loader = DataLoader(
            Subset(train_ds_opt, fold.train_indices),
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
            num_workers=0,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(val_ds_opt, fold.val_indices),
            batch_size=batch_size,
            num_workers=0,
            pin_memory=True
        )
        
        model = MTDE_Net(dropout=dropout).to(device)
        crit = SqrtScaledMSELoss(scale=None)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        epochs = 50
        patience = 8
        best_fold_mae = float("inf")
        no_improvement = 0
        
        for ep in range(epochs):
            model.train()
            for x_img, x_tab, y in train_loader:
                x_img, x_tab, y = x_img.to(device), x_tab.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                loss = crit(model(x_img, x_tab), y)
                loss.backward()
                opt.step()
                
            model.eval()
            val_m = eval_mtde_net_metrics(model, val_loader, device, scale=time_scale)
            v_mae = val_m["mae"]
            if v_mae < best_fold_mae - 0.5:
                best_fold_mae = v_mae
                no_improvement = 0
            else:
                no_improvement += 1
                
            if no_improvement >= patience:
                break
                
        fold_maes.append(best_fold_mae)
        trial.report(float(np.mean(fold_maes)), step=fold_idx)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return float(np.mean(fold_maes))

In [4]:
study_name_mtde = "mtde_net_optimization_hybrid_v1"
storage_name = "sqlite:///optuna_study.db"

study_mtde = optuna.create_study(
    study_name=study_name_mtde,
    storage=storage_name,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

# Ejecutar optimización para MTDE-Net (ejemplo: 20 trials)
study_mtde.optimize(objective_mtde_net, n_trials=20)

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS (MTDE-Net) ===")
print(study_mtde.best_params)
print(f"Mejor MAE CV: {study_mtde.best_value:.2f}s")

[I 2026-06-29 18:56:25,933] A new study created in RDB with name: mtde_net_optimization_hybrid_v1
[I 2026-06-29 19:14:46,938] Trial 0 finished with value: 39.36587562561035 and parameters: {'lr': 0.0005085169637254012, 'weight_decay': 0.0013418922681654594, 'batch_size': 16, 'dropout': 0.11582908187752085}. Best is trial 0 with value: 39.36587562561035.
[I 2026-06-29 19:30:57,957] Trial 1 finished with value: 44.711181640625 and parameters: {'lr': 5.161825150601387e-05, 'weight_decay': 0.0008047714731199909, 'batch_size': 32, 'dropout': 0.2015700353753564}. Best is trial 0 with value: 39.36587562561035.
[I 2026-06-29 19:37:59,825] Trial 2 finished with value: 49.262823486328124 and parameters: {'lr': 0.0019315377262791437, 'weight_decay': 2.1403250217665416e-05, 'batch_size': 64, 'dropout': 0.2795008953813781}. Best is trial 0 with value: 39.36587562561035.
[I 2026-06-29 19:49:21,091] Trial 3 finished with value: 66.9752326965332 and parameters: {'lr': 6.84619106137298e-05, 'weight_dec


=== MEJORES HIPERPARÁMETROS ENCONTRADOS (MTDE-Net) ===
{'lr': 0.00019838749758029734, 'weight_decay': 0.0031990339484422084, 'batch_size': 16, 'dropout': 0.21668657928816884}
Mejor MAE CV: 37.12s


## 3. Optimización para CNN-LSTM (Modelo Secuencial)

In [5]:
def objective_cnn_lstm(trial):
    set_seed(42)
    
    # Espacio de Búsqueda para CNN-LSTM
    lr = trial.suggest_float("lr", 5e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lstm_hidden_dim = trial.suggest_categorical("lstm_hidden_dim", [64, 128])
    seq_len = trial.suggest_categorical("seq_len", [3, 5, 8])
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    time_scale = 30.0
    df = train_ds_opt.df
    fold_maes = []
    
    for fold_idx, fold in enumerate(plan_opt.folds):
        train_seq_ids = sequence_ids_for_subjects(df, fold.train_subjects)
        val_seq_ids = sequence_ids_for_subjects(df, fold.val_subjects)
        
        train_ds = ThermalSequenceDataset(
            metadata_csv=metadata_path,
            is_train=True,
            min_time_s=0.0,
            seq_len=seq_len,
            sequence_ids=train_seq_ids
        )
        val_ds = ThermalSequenceDataset(
            metadata_csv=metadata_path,
            is_train=False,
            min_time_s=0.0,
            seq_len=seq_len,
            sequence_ids=val_seq_ids
        )
        train_ds.root = Path("../processed_data")
        val_ds.root = Path("../processed_data")
        
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
            num_workers=0,
            pin_memory=True
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            num_workers=0,
            pin_memory=True
        )
        
        model = ThermalCNNLSTM(
            lstm_hidden_dim=lstm_hidden_dim,
            dropout=dropout
        ).to(device)
        
        crit = SqrtScaledMSELoss(scale=time_scale)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        epochs = 30  # Menos épocas para acelerar la búsqueda
        patience = 6
        best_fold_mae = float("inf")
        no_improvement = 0
        
        for ep in range(epochs):
            model.train()
            for x_seq, y in train_loader:
                x_seq, y = x_seq.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                loss = crit(model(x_seq), y)
                loss.backward()
                opt.step()
                
            model.eval()
            val_m = eval_cnn_lstm_metrics(model, val_loader, device)
            v_mae = val_m["mae"]
            if v_mae < best_fold_mae - 0.5:
                best_fold_mae = v_mae
                no_improvement = 0
            else:
                no_improvement += 1
                
            if no_improvement >= patience:
                break
                
        fold_maes.append(best_fold_mae)
        trial.report(float(np.mean(fold_maes)), step=fold_idx)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    return float(np.mean(fold_maes))

In [6]:
study_name_lstm = "cnn_lstm_optimization_hybrid_v1"
storage_name = "sqlite:///optuna_study.db"

study_lstm = optuna.create_study(
    study_name=study_name_lstm,
    storage=storage_name,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

# Ejecutar optimización para CNN-LSTM (ejemplo: 20 trials)
study_lstm.optimize(objective_cnn_lstm, n_trials=20)

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS (CNN-LSTM) ===")
print(study_lstm.best_params)
print(f"Mejor MAE CV: {study_lstm.best_value:.2f}s")

[I 2026-07-01 00:25:35,695] Using an existing study with name 'cnn_lstm_optimization_hybrid_v1' instead of creating a new one.
[I 2026-07-01 00:48:50,601] Trial 3 finished with value: 44.470085525512694 and parameters: {'lr': 9.848611671197905e-05, 'weight_decay': 2.062532950725845e-05, 'batch_size': 8, 'dropout': 0.3444139526079738, 'lstm_hidden_dim': 64, 'seq_len': 3}. Best is trial 3 with value: 44.470085525512694.
[I 2026-07-01 01:15:10,459] Trial 4 finished with value: 39.537945938110354 and parameters: {'lr': 0.0009369850216370142, 'weight_decay': 5.357255367896619e-05, 'batch_size': 8, 'dropout': 0.37675212140698333, 'lstm_hidden_dim': 128, 'seq_len': 5}. Best is trial 4 with value: 39.537945938110354.
[I 2026-07-01 01:41:23,783] Trial 5 finished with value: 39.72043075561523 and parameters: {'lr': 0.0003364822976391303, 'weight_decay': 1.2903508715294726e-05, 'batch_size': 8, 'dropout': 0.2995979171005052, 'lstm_hidden_dim': 128, 'seq_len': 5}. Best is trial 4 with value: 39.53


=== MEJORES HIPERPARÁMETROS ENCONTRADOS (CNN-LSTM) ===
{'lr': 0.0007706419796586753, 'weight_decay': 0.00023859221722069722, 'batch_size': 8, 'dropout': 0.24024397301432543, 'lstm_hidden_dim': 128, 'seq_len': 3}
Mejor MAE CV: 35.78s
